# GA 10

In [1]:
!pip install -q "transformers>=4.44.0" "accelerate>=0.31.0" "peft>=0.13.0" datasets evaluate scikit-learn pandas bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.3 MB/s eta 0:00:00


In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import DatasetDict, load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, TaskType
import torch

⚙️  Running in WANDB offline mode


In [3]:
df = pd.read_csv("/content/data.csv")

species_list = sorted(df["species"].unique())
label2id = {s:i for i,s in enumerate(species_list)}
id2label = {i:s for s,i in label2id.items()}

df["label"] = df["species"].map(label2id)

df.head()

,sepal_length,sepal_width,petal_length,petal_width,species,label
0,5.8,4.0,1.2,0.2,setosa,0
1,5.7,4.4,1.5,0.4,setosa,0
2,5.4,3.9,1.3,0.4,setosa,0
3,5.1,3.5,1.4,0.3,setosa,0
4,5.7,3.8,1.7,0.3,setosa,0


In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.10, random_state=42, stratify=df["species"]
)

# For v1

In [5]:
def numeric_to_text(row_or_dict):
    # row_or_dict can be a pandas row or a simple dict with the right keys
    sl = row_or_dict["sepal_length"]
    sw = row_or_dict["sepal_width"]
    pl = row_or_dict["petal_length"]
    pw = row_or_dict["petal_width"]
    return (
        "Classify the iris flower species from numeric measurements.\n"
        f"Sepal length: {sl}\n"
        f"Sepal width: {sw}\n"
        f"Petal length: {pl}\n"
        f"Petal width: {pw}\n"
        "Answer only with the species name."
    )

train_df["text_numeric"] = train_df.apply(numeric_to_text, axis=1)
test_df["text_numeric"]  = test_df.apply(numeric_to_text, axis=1)

# For v2

In [6]:
bin_edges = {}
for col in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    q1 = df[col].quantile(0.33)
    q2 = df[col].quantile(0.66)
    bin_edges[col] = (q1, q2)

def bin_value(x, q1, q2):
    if x <= q1: return "low"
    if x <= q2: return "medium"
    return "high"
def binned_description(row, edges):
    sl = bin_value(row["sepal_length"], *edges["sepal_length"])
    sw = bin_value(row["sepal_width"], *edges["sepal_width"])
    pl = bin_value(row["petal_length"], *edges["petal_length"])
    pw = bin_value(row["petal_width"], *edges["petal_width"])

    return (
        f"The iris flower has a {sl} sepal length, "
        f"a {sw} sepal width, "
        f"a {pl} petal length, "
        f"and a {pw} petal width. "
        f"Predict the species."
    )
train_df["text_binned"] = train_df.apply(lambda r: binned_description(r, bin_edges), axis=1)
test_df["text_binned"]  = test_df.apply(lambda r: binned_description(r, bin_edges), axis=1)


In [7]:
def save_jsonl(df, field, filename):
    with open(filename, "w") as f:
        for _, row in df.iterrows():
            f.write(json.dumps({"text": row[field], "label": int(row["label"])}) + "\n")

save_jsonl(train_df, "text_numeric", "train_numeric.jsonl")
save_jsonl(test_df,  "text_numeric", "test_numeric.jsonl")

save_jsonl(train_df, "text_binned", "train_binned.jsonl")
save_jsonl(test_df,  "text_binned", "test_binned.jsonl")

In [8]:
!head -n 5 /content/train_numeric.jsonl  /content/train_binned.jsonl

==> /content/train_numeric.jsonl <==
{"text": "Classify the iris flower species from numeric measurements.\nSepal length: 5.7\nSepal width: 4.4\nPetal length: 1.5\nPetal width: 0.4\nAnswer only with the species name.", "label": 0}
{"text": "Classify the iris flower species from numeric measurements.\nSepal length: 6.7\nSepal width: 3.3\nPetal length: 5.7\nPetal width: 2.5\nAnswer only with the species name.", "label": 2}
{"text": "Classify the iris flower species from numeric measurements.\nSepal length: 6.6\nSepal width: 3.0\nPetal length: 4.4\nPetal width: 1.4\nAnswer only with the species name.", "label": 1}
{"text": "Classify the iris flower species from numeric measurements.\nSepal length: 6.9\nSepal width: 3.1\nPetal length: 5.4\nPetal width: 2.1\nAnswer only with the species name.", "label": 2}
{"text": "Classify the iris flower species from numeric measurements.\nSepal length: 6.3\nSepal width: 2.5\nPetal length: 4.9\nPetal width: 1.5\nAnswer only with the species name.", "labe

In [9]:
datasets_numeric = DatasetDict({
    "train": load_dataset("json", data_files="train_numeric.jsonl")["train"],
    "test":  load_dataset("json", data_files="test_numeric.jsonl")["train"],
})

datasets_binned = DatasetDict({
    "train": load_dataset("json", data_files="train_binned.jsonl")["train"],
    "test":  load_dataset("json", data_files="test_binned.jsonl")["train"],
})

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [44]:
bf16_available = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16_available else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype
)

In [45]:
def load_model_with_lora():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model.config.pad_token_id = tokenizer.pad_token_id

    # Memory saving
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    lora_cfg = LoraConfig(
      task_type=TaskType.SEQ_CLS,
      r=16,
      lora_alpha=32,
      lora_dropout=0.05,
      bias="none"
    )


    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model

In [25]:
MAX_LEN = 64

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

encoded_numeric = datasets_numeric.map(tokenize, batched=True)
encoded_binned  = datasets_binned.map(tokenize, batched=True)

encoded_numeric = encoded_numeric.remove_columns(["text"])
encoded_binned  = encoded_binned.remove_columns(["text"])

encoded_numeric.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
encoded_binned.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

In [14]:
def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [59]:
def train_model(model,encoded_dataset, tag):

    training_args = TrainingArguments(
    output_dir=f"./gemma2_{tag}",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-4,
    num_train_epochs=1,
    weight_decay=0.0,
    max_grad_norm=1.0,

    eval_strategy="epoch",   # evaluation once per epoch (small table)
    logging_strategy="epoch",
    save_strategy="epoch",

    report_to="none",
)



    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    print(f"\n===== TRAINING {tag.upper()} MODEL =====")
    trainer.train()

    print("\n=== FINAL TEST EVALUATION ===")
    test_metrics = trainer.evaluate(encoded_dataset["test"])
    print(test_metrics)

    return trainer, test_metrics


In [60]:
model_numeric = load_model_with_lora()
trainer_numeric, metrics_numeric = train_model(model_numeric, encoded_numeric, "numeric")

model_binned = load_model_with_lora()
trainer_binned, metrics_binned = train_model(model_binned, encoded_binned, "binned")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at google/gemma-2-2b-it and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2956002797.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


trainable params: 3,201,792 || all params: 2,617,550,592 || trainable%: 0.1223

===== TRAINING NUMERIC MODEL =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.650800,1.213157,0.363636,0.177778



=== FINAL TEST EVALUATION ===


...

Some weights of Gemma2ForSequenceClassification were not initialized from the model checkpoint at google/gemma-2-2b-it and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2956002797.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


trainable params: 3,201,792 || all params: 2,617,550,592 || trainable%: 0.1223

===== TRAINING BINNED MODEL =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1141.405600,56.863636,0.454545,0.323810



=== FINAL TEST EVALUATION ===


{'eval_loss': 56.8636360168457, 'eval_accuracy': 0.45454545454545453, 'eval_macro_f1': 0.3238095238095238, 'eval_runtime': 2.7385, 'eval_samples_per_second': 4.017, 'eval_steps_per_second': 4.017, 'epoch': 1.0}


In [62]:
(0.45-0.36)/0.36 * 100

25.000000000000007

In [71]:
def predict_numeric(sl, sw, pl, pw):
    row = {
        "sepal_length": sl,
        "sepal_width": sw,
        "petal_length": pl,
        "petal_width": pw
    }
    text = numeric_to_text(row)
    inputs = tokenizer(text, return_tensors="pt").to(model_numeric.device)

    with torch.no_grad():
        logits = model_numeric(**inputs).logits

    pred_id = logits.argmax(-1).item()
    return id2label[pred_id], text


def predict_binned(sl, sw, pl, pw):
    row = {
        "sepal_length": sl,
        "sepal_width": sw,
        "petal_length": pl,
        "petal_width": pw
    }
    text = binned_description(row, bin_edges)
    inputs = tokenizer(text, return_tensors="pt").to(model_binned.device)

    with torch.no_grad():
        logits = model_binned(**inputs).logits

    pred_id = logits.argmax(-1).item()
    return id2label[pred_id], text


def show_all_exact(sl, sw, pl, pw):
    # find exact row
    mask = (
        (df["sepal_length"] == sl) &
        (df["sepal_width"] == sw) &
        (df["petal_length"] == pl) &
        (df["petal_width"] == pw)
    )

    if not mask.any():
        raise ValueError(
            f"No exact match found for:\n"
            f"SL={sl}, SW={sw}, PL={pl}, PW={pw}\n"
            f"Make sure values come from the CSV exactly."
        )

    row = df[mask].iloc[0]
    true_label = row["species"]

    # Predictions from both models
    pred_num, text_num = predict_numeric(sl, sw, pl, pw)
    pred_bin, text_bin = predict_binned(sl, sw, pl, pw)

    print("🔵 INPUT VALUES (EXACT FROM DATASET)\n==============================")
    print(f" Sepal Len={sl}, Sepal Wid={sw}, Petal Len={pl}, Petal Wid={pw}\n")

    print("🟢 ORIGINAL SPECIES (CSV)\n============================")
    print(f"   {true_label}\n")

    print("🟣 NUMERIC MODEL PREDICTION\n==========================")
    print(f"   Prediction: {pred_num}")
    print("   Prompt:")
    print("   " + text_num.replace("\n", "\n   "), "\n")

    print("🟡 BINNED MODEL PREDICTION\n===========================")
    print(f"   Prediction: {pred_bin}")
    print("   Prompt:")
    print("   " + text_bin.replace("\n", "\n   "), "\n")


In [72]:
show_all_exact(5.7, 4.4, 1.5, 0.4)

🔵 INPUT VALUES (EXACT FROM DATASET)
 Sepal Len=5.7, Sepal Wid=4.4, Petal Len=1.5, Petal Wid=0.4

🟢 ORIGINAL SPECIES (CSV)
   setosa

🟣 NUMERIC MODEL PREDICTION
   Prediction: virginica
   Prompt:
   Classify the iris flower species from numeric measurements.
   Sepal length: 5.7
   Sepal width: 4.4
   Petal length: 1.5
   Petal width: 0.4
   Answer only with the species name. 

🟡 BINNED MODEL PREDICTION
   Prediction: setosa
   Prompt:
   The iris flower has a medium sepal length, a high sepal width, a low petal length, and a low petal width. Predict the species. 

